# Exp10: Gemma-2 IOI, per-language representation test
Question: does IOI use shared representations across EN/FR/Yoruba, and does separation grow with scale?
**Run with SMOKE=True first (2B, tiny).** Only then set SMOKE=False.

Worlds (cell 7 + 8 read these):
| World | Head ablation (EN heads) -> FR | Subspace ablation (EN subspace) -> FR |
|---|---|---|
| Fully shared | drops | drops |
| Shared heads, separate subspaces | drops | no drop |
| Duplicated pathways | no drop | no drop |

Caveats: Yoruba templates are mine, have a native speaker check them. Subspace = top-K PCs of END-position residual (variance, not IOI-specific), so read it against the random-subspace control.

In [3]:
import os
os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN", "<REDACTED>")   # your token

In [4]:
import os, socket
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))
try:
    socket.create_connection(("huggingface.co", 443), timeout=5); print("network: OK")
except Exception as e:
    print("network: BLOCKED", e)
from huggingface_hub import whoami, hf_hub_download
try: print(whoami(token=os.environ.get("HF_TOKEN"))["name"])
except Exception as e: print("auth:", e)
try: hf_hub_download("google/gemma-2-2b", "config.json", token=os.environ.get("HF_TOKEN")); print("gated access: OK")
except Exception as e: print("gated access:", type(e).__name__, str(e)[:200])

HF_TOKEN set: True
network: BLOCKED [Errno 101] Network is unreachable
Sat1501
gated access: OK


In [5]:
import os
GPU_ID = "1"   # your sentinel; set None to use all visible GPUs
if GPU_ID is not None: os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID   # must precede torch import
import json, gc, random
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

SMOKE = True          # True: 2B only, tiny. Run first.
HF_TOKEN = os.environ.get("HF_TOKEN")   # Gemma is gated
MODELS = ["google/gemma-2-2b", "google/gemma-2-9b", "google/gemma-2-27b"]
LANGS = ["en", "fr", "yo"]
LAYER_FRACS = [0.33, 0.5, 0.67, 0.85]   # geometry (mid layers are the ones to weigh)
CAUSAL_FRAC = 0.5
K_ANGLE = 4
K_SUB = 8
GATE_ACC = 0.75       # lang dropped for a model if IOI acc below this
RESULTS_DIR = "exp10_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

if SMOKE:
    MODELS = MODELS[:1]; N_PROMPTS = 60; SEEDS = [0]; K_HEADS = 5; SCAN_N = 8
else:
    N_PROMPTS = 420; SEEDS = [0, 1, 2]; K_HEADS = 10; SCAN_N = 16
assert torch.cuda.is_available(), "no CUDA visible: check GPU_ID"
props = torch.cuda.get_device_properties(0)
print("SMOKE" if SMOKE else "FULL", MODELS, torch.cuda.device_count(), "GPU(s):", props.name, round(props.total_memory/2**30), "GiB")
print("27B bf16 needs ~54 GiB + activations; 9B ~18; 2B ~5")

SMOKE ['google/gemma-2-2b'] 1 GPU(s): NVIDIA H200 NVL 140 GiB
27B bf16 needs ~54 GiB + activations; 9B ~18; 2B ~5


In [6]:
TEMPLATES = {
 "en": ["When {N1} and {N2} went to the store, {S} gave a drink to",
        "Then, {N1} and {N2} had a lot of fun at the school. {S} gave a ball to",
        "After {N1} and {N2} went to the park, {S} handed a book to",
        "Yesterday {N1} and {N2} visited the office, and {S} sent a letter to",
        "While {N1} and {N2} were at the market, {S} offered some fruit to",
        "Today {N1} and {N2} met in the garden. Afterwards, {S} showed the photos to"],
 "fr": ["Quand {N1} et {N2} sont allés au marché, {S} a donné une pomme à",
        "Ensuite, {N1} et {N2} se sont bien amusés à l'école. {S} a donné un ballon à",
        "Après que {N1} et {N2} sont allés au parc, {S} a prêté un livre à",
        "Hier, {N1} et {N2} ont visité le bureau, et {S} a envoyé une lettre à",
        "Pendant que {N1} et {N2} étaient au marché, {S} a offert des fruits à",
        "Aujourd'hui, {N1} et {N2} se sont retrouvés dans le jardin. Ensuite, {S} a montré les photos à"],
 "yo": ["Nígbà tí {N1} àti {N2} lọ sí ọjà, {S} ra ẹ̀bùn kan fún",
        "Lẹ́yìn náà, {N1} àti {N2} gbádùn ní ilé-ẹ̀kọ́. {S} fi bọ́ọ̀lù kan fún",
        "Lẹ́yìn tí {N1} àti {N2} lọ sí ọgbà, {S} yá ìwé kan fún",
        "Lánàá {N1} àti {N2} ṣàbẹ̀wò sí ọ́fíìsì, {S} sì fi lẹ́tà kan ranṣẹ́ sí",
        "Nígbà tí {N1} àti {N2} wà ní ọjà, {S} mú èso wá fún",
        "Lónìí {N1} àti {N2} pàdé ní ọgbà. Lẹ́yìn náà, {S} fi àwòrán hàn fún"],
}
NAMES = {
 "en": "James John Robert Michael William David Richard Thomas Mary Patricia Jennifer Linda Elizabeth Barbara Susan Jessica Sarah Karen Daniel Mark Paul Steven Kevin Brian George Anna Laura Emma Alice Julia Peter Henry Frank Kate Lucy Ruth Helen Carol Diane Grace Rose Edward Andrew Joseph Charles".split(),
 "fr": "Jean Pierre Louis Jacques Michel Henri Marie Claire Sophie Camille Lucie Chloé Hugo Lucas Antoine Nicolas Alain Luc Éric Anne Isabelle Nathalie Sylvie Céline Julien Mathieu Olivier Bruno Léa Manon Étienne Gérard Vincent Émile Alexandre Margot Inès Mathilde Clémence Aurélie Baptiste Guillaume".split(),
 "yo": "Ayo Tunde Femi Bola Kemi Dayo Tola Segun Yemi Wale Funmi Ade Bisi Folake Kunle Lola Sade Tayo Seyi Bimpe Dupe Yinka Kola Ladi Remi Toyin Jide Gbenga Ife Tobi Temi Wumi Niyi Shola Sola Lekan Dele Kayode Bukola Ronke Shade Bayo Biodun Taiwo Kehinde Lanre".split(),
}
NAMES["fr"] = [n for n in NAMES["fr"] if n not in set(NAMES["en"])]   # keep lexically disjoint from EN

In [10]:
def single_token_names(tok, names):
    out, nid, seen = [], {}, set()
    for n in names:
        ids = tok.encode("a " + n, add_special_tokens=False)
        if len(ids) == 2 and ids[1] not in seen:
            out.append(n); nid[n] = ids[1]; seen.add(ids[1])
    return out, nid

def build(tok, lang, tmpl_ids, names, nid, n, rng):
    per = max(1, n // len(tmpl_ids)); batches = []
    for t in tmpl_ids:
        rows = []
        for _ in range(per):
            io, s = rng.sample(names, 2)
            n1, n2 = (io, s) if rng.random() < 0.5 else (s, io)
            ids = tok(TEMPLATES[lang][t].format(N1=n1, N2=n2, S=s))["input_ids"]
            rows.append((ids, nid[io], nid[s]))
        lens = [len(r[0]) for r in rows]
        L = max(set(lens), key=lens.count)
        rows = [r for r in rows if len(r[0]) == L]
        batches.append(dict(
            x=torch.tensor([r[0] for r in rows]),
            io=torch.tensor([r[1] for r in rows]), s=torch.tensor([r[2] for r in rows]),
            pIO=torch.tensor([r[0].index(r[1]) for r in rows]),
            pS2=torch.tensor([len(r[0]) - 1 - r[0][::-1].index(r[2]) for r in rows])))
    return batches

def slice_b(b, n): return {k: v[:n] for k, v in b.items()}

class Hooks:
    def __init__(self): self.h = []
    def add(self, m, kind, fn): self.h.append(getattr(m, kind)(fn))
    def __enter__(self): return self
    def __exit__(self, *a): [h.remove() for h in self.h]

def subspace(X, k):
    Xc = X - X.mean(0)
    return torch.linalg.svd(Xc, full_matrices=False)[2][:k]

def angle(X, Y, k):
    s = torch.linalg.svdvals(subspace(X, k) @ subspace(Y, k).T).clamp(-1, 1)
    return torch.rad2deg(torch.arccos(s)).mean().item()

In [11]:
@torch.no_grad()
def fwd(model, b, hs_layers=(), bs=32):
    dev = model.model.embed_tokens.weight.device
    cap = getattr(model.config, "final_logit_softcapping", None)
    lds, accs = [], []
    hs = {l: {"IO": [], "S2": [], "END": []} for l in hs_layers}
    for i in range(0, len(b["x"]), bs):
        x = b["x"][i:i+bs].to(dev)
        out = model.model(input_ids=x, output_hidden_states=bool(hs_layers))
        lg = model.lm_head(out.last_hidden_state[:, -1])
        if cap: lg = torch.tanh(lg / cap) * cap
        lg = lg.float(); ar = torch.arange(len(x), device=lg.device)
        d = lg[ar, b["io"][i:i+bs].to(lg.device)] - lg[ar, b["s"][i:i+bs].to(lg.device)]
        lds.append(d.cpu()); accs.append((d > 0).float().cpu())
        for l in hs_layers:
            h = out.hidden_states[l]; a2 = torch.arange(len(x), device=h.device)
            hs[l]["IO"].append(h[a2, b["pIO"][i:i+bs].to(h.device)].float().cpu())
            hs[l]["S2"].append(h[a2, b["pS2"][i:i+bs].to(h.device)].float().cpu())
            hs[l]["END"].append(h[:, -1].float().cpu())
    return torch.cat(lds), torch.cat(accs), {l: {p: torch.cat(v) for p, v in d.items()} for l, d in hs.items()}

def ev(model, batches, hs_layers=()):
    rs = [fwd(model, b, hs_layers) for b in batches]
    ld = torch.cat([r[0] for r in rs]); acc = torch.cat([r[1] for r in rs])
    hs = {l: {p: torch.cat([r[2][l][p] for r in rs]) for p in ("IO", "S2", "END")} for l in hs_layers}
    return ld, acc, hs

def head_hook(hs_idx, HD):
    def fn(mod, args):
        x = args[0].clone()
        for h in hs_idx: x[..., h*HD:(h+1)*HD] = 0
        return (x,) + tuple(args[1:])
    return fn

def ev_heads(model, batches, heads):   # heads: list of (layer, head)
    HD = model.config.head_dim; by = {}
    for l, h in heads: by.setdefault(l, []).append(h)
    with Hooks() as H:
        for l, hh in by.items():
            H.add(model.model.layers[l].self_attn.o_proj, "register_forward_pre_hook", head_hook(hh, HD))
        return ev(model, batches)[0]

def sub_hook(V, mu):
    def fn(mod, args, out):
        tup = isinstance(out, tuple); h = (out[0] if tup else out).clone()
        last = h[:, -1].float(); Vd = V.to(h.device); m = mu.to(h.device)
        h[:, -1] = (last - ((last - m) @ Vd.T) @ Vd).to(h.dtype)
        return (h,) + tuple(out[1:]) if tup else h
    return fn

def ev_sub(model, batches, layer_idx, V, mu):
    with Hooks() as H:
        H.add(model.model.layers[layer_idx], "register_forward_hook", sub_hook(V, mu))
        return ev(model, batches)[0]

In [8]:
def run_model(name):
    path = f"{RESULTS_DIR}/{name.split('/')[-1]}{'_smoke' if SMOKE else ''}.json"
    if os.path.exists(path):
        print("cached", path); return json.load(open(path))
    tok = AutoTokenizer.from_pretrained(name, token=HF_TOKEN)
    model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.bfloat16, device_map="auto",
                                                 attn_implementation="eager", token=HF_TOKEN).eval()
    NL, NH = model.config.num_hidden_layers, model.config.num_attention_heads
    res = {"model": name, "gate": {}, "angles": [], "heads": {}, "subspace": {}}

    names, nid = {}, {}
    for L in LANGS:
        names[L], nid[L] = single_token_names(tok, NAMES[L])
        print(L, len(names[L]), "single-token names")
        if len(names[L]) < 8: print("  WARNING few names for", L)
    nid_all = {n: i for L in LANGS for n, i in nid[L].items()}
    en = names["en"][:]; random.Random(123).shuffle(en)
    nA, nB = en[:len(en)//2], en[len(en)//2:]

    def conds(seed):
        r = random.Random(seed); T = list(range(6))
        return {"EN": build(tok, "en", T, names["en"], nid_all, N_PROMPTS, r),
                "FR": build(tok, "fr", T, names["fr"], nid_all, N_PROMPTS, r),
                "YO": build(tok, "yo", T, names["yo"], nid_all, N_PROMPTS, r),
                "EN_T1": build(tok, "en", [0, 1, 2], names["en"], nid_all, N_PROMPTS, r),
                "EN_T2": build(tok, "en", [3, 4, 5], names["en"], nid_all, N_PROMPTS, r),
                "EN_NA": build(tok, "en", T, nA, nid_all, N_PROMPTS, r),
                "EN_NB": build(tok, "en", T, nB, nid_all, N_PROMPTS, r)}

    # 1. gate
    C0 = conds(SEEDS[0]); ok = {}
    for L in LANGS:
        ld, acc, _ = ev(model, C0[L.upper()])
        res["gate"][L] = {"acc": acc.mean().item(), "logit_diff": ld.mean().item(), "n": len(ld)}
        ok[L] = acc.mean().item() >= GATE_ACC
    print("GATE", res["gate"], "ok:", ok)

    # 2. geometry (Chang-style: mean-centered, fixed k, within-model)
    layers = [int(f * NL) for f in LAYER_FRACS]
    for seed in SEEDS:
        C = C0 if seed == SEEDS[0] else conds(seed)
        H = {k: ev(model, v, layers)[2] for k, v in C.items()}
        for f, l in zip(LAYER_FRACS, layers):
            for p in ("IO", "S2", "END"):
                row = {"seed": seed, "frac": f, "layer": l, "pos": p,
                       "floor_template": angle(H["EN_T1"][l][p], H["EN_T2"][l][p], K_ANGLE),
                       "floor_nameset": angle(H["EN_NA"][l][p], H["EN_NB"][l][p], K_ANGLE)}
                for L in ("fr", "yo"):
                    row["EN-" + L.upper()] = angle(H["EN"][l][p], H[L.upper()][l][p], K_ANGLE) if ok[L] else None
                res["angles"].append(row)
        del H

    # 3a. head ablation: rank EN heads by zero-ablation drop, then ablate top-K in each language
    sb = slice_b(C0["EN"][0], SCAN_N); base = ev(model, [sb])[0].mean().item()
    score = np.zeros((NL, NH))
    for l in range(NL):
        for h in range(NH):
            score[l, h] = base - ev_heads(model, [sb], [(l, h)]).mean().item()
    flat = sorted(((score[l, h], l, h) for l in range(NL) for h in range(NH)), reverse=True)
    top = [(l, h) for _, l, h in flat[:K_HEADS]]
    rr = random.Random(0); allh = [(l, h) for l in range(NL) for h in range(NH)]
    rands = [rr.sample(allh, K_HEADS) for _ in range(3)]
    res["heads"]["top"] = top
    for L in LANGS:
        if not ok[L]: continue
        clean = ev(model, C0[L.upper()])[0].mean().item()
        ab = ev_heads(model, C0[L.upper()], top).mean().item()
        rd = float(np.mean([ev_heads(model, C0[L.upper()], r).mean().item() for r in rands]))
        res["heads"][L] = {"clean": clean, "top_ablated": ab, "random_ablated": rd,
                           "retained_top": ab / clean, "retained_rand": rd / clean}

    # 3b. subspace ablation matrix: mean-ablate src-language top-K subspace in tgt language
    Lh = int(CAUSAL_FRAC * NL)
    hsl = {L: ev(model, C0[L.upper()], [Lh])[2][Lh]["END"] for L in LANGS if ok[L]}
    g = torch.Generator().manual_seed(0)
    Vr = torch.linalg.qr(torch.randn(hsl["en"].shape[1], K_SUB, generator=g))[0].T
    for src in hsl:
        V = subspace(hsl[src], K_SUB); res["subspace"][src] = {}
        for tgt in hsl:
            clean = ev(model, C0[tgt.upper()])[0].mean().item()
            mu = hsl[tgt].mean(0)
            a = ev_sub(model, C0[tgt.upper()], Lh - 1, V, mu).mean().item()
            r = ev_sub(model, C0[tgt.upper()], Lh - 1, Vr, mu).mean().item()
            res["subspace"][src][tgt] = {"retained": a / clean, "retained_rand": r / clean}

    json.dump(res, open(path, "w"), indent=1)
    del model; gc.collect(); torch.cuda.empty_cache()
    return res

In [12]:
ALL = {m: run_model(m) for m in MODELS}

for m, r in ALL.items():
    print("\n=====", m)
    print("GATE:", {k: round(v["acc"], 3) for k, v in r["gate"].items()})
    df = pd.DataFrame(r["angles"]).groupby(["frac", "pos"]).agg(["mean", "std"]).drop(columns=["seed", "layer"], level=0)
    print(df.round(1).to_string())
    print("\nHEAD ABLATION (retained logit-diff fraction; top EN heads vs random):")
    print(pd.DataFrame({k: v for k, v in r["heads"].items() if k != "top"}).T.round(3).to_string())
    print("\nSUBSPACE ABLATION (rows=src subspace, cols=tgt lang; retained fraction):")
    print(pd.DataFrame({s: {t: x["retained"] for t, x in d.items()} for s, d in r["subspace"].items()}).T.round(3).to_string())
    print("random-subspace control:")
    print(pd.DataFrame({s: {t: x["retained_rand"] for t, x in d.items()} for s, d in r["subspace"].items()}).T.round(3).to_string())

Loading weights: 100%|██████████| 288/288 [00:00<00:00, 771.56it/s]


en 45 single-token names
fr 34 single-token names
yo 12 single-token names
GATE {'en': {'acc': 1.0, 'logit_diff': 4.116666793823242, 'n': 60}, 'fr': {'acc': 1.0, 'logit_diff': 3.9166667461395264, 'n': 60}, 'yo': {'acc': 0.8500000238418579, 'logit_diff': 3.9385416507720947, 'n': 60}} ok: {'en': True, 'fr': True, 'yo': True}

===== google/gemma-2-2b
GATE: {'en': 1.0, 'fr': 1.0, 'yo': 0.85}
         floor_template     floor_nameset     EN-FR     EN-YO    
                   mean std          mean std  mean std  mean std
frac pos                                                         
0.33 END           64.0 NaN           4.3 NaN  52.6 NaN  86.0 NaN
     IO            42.1 NaN          62.0 NaN  64.5 NaN  81.8 NaN
     S2            58.9 NaN          67.5 NaN  70.7 NaN  83.3 NaN
0.50 END           71.2 NaN           6.6 NaN  48.9 NaN  87.1 NaN
     IO            46.5 NaN          55.0 NaN  55.0 NaN  76.6 NaN
     S2            54.3 NaN          39.7 NaN  44.2 NaN  78.8 NaN
0.67 END       